# Charge Category Classification

Goal: take the raw `Charge Type` / `Charge Description` labels from `trackingdetail Table.csv` (1,890 / 2,968 distinct raw values in the sample data) and roll them up into a small taxonomy: **5-7 major categories, each with up to 10 subcategories**.

The taxonomy itself (`TAXONOMY`, `PRIORITY_OVERRIDES`, `rule_classify_row`) lives in `taxonomy.py`, imported by both this notebook and `charge_categorization_check.ipynb`, so the two can never drift out of sync (they used to be copy-pasted into each notebook separately).

**Approach**: rule-based keyword/regex classification, with a lightweight TF-IDF fuzzy-match fallback for anything the rules miss. No transformer/embedding model is used, on purpose:

- The vocabulary here is a *finite domain vocabulary* (carrier billing terms across a handful of languages), not open-ended natural language — keyword rules cover the large majority of it directly.
- Rule-based mapping is fully auditable: for any classified row you can point to the exact pattern that matched it. That matters for billing/finance data.
- Classification runs on the **unique (Charge Type, Charge Description) combinations**, not per row — there are only a few thousand of those even in the sample. This is what makes the approach reusable: production data with millions of rows still only has a few thousand distinct label combinations, so the same mapping table (or the same classifier re-run on new unique labels) scales without change.
- No heavy dependencies (torch / sentence-transformers) — just `scikit-learn`, which is already installed.

If, once applied to real company data, coverage from rules + TF-IDF fallback isn't high enough, an embedding-based clustering pass is a reasonable upgrade path — but it's not needed to get started.

In [1]:
import re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)
ENC = "cp1252"

## Load charge-level data

In [2]:
detail = pd.read_csv(
    "Database report1788285968903_trackingdetail Table.csv",
    encoding=ENC,
    usecols=["Charge Type", "Charge Description", "Charge Value"],
    low_memory=False,
)
print(detail.shape)
detail.head()

(256764, 3)


,Charge Description,Charge Type,Charge Value
0,SERVICE RETURN TO WAREHOUSE,SERVICE RETURN TO WAREHOUSE,246.00
1,Transportation Charge,Transportation Charge,22.21
2,Discount,Discount,-9.66
3,ACTUATORS 70 304 MIN,Freight,158.29
4,Earned Discount,Earned Discount,-4.74


## Taxonomy: 7 substantive major categories (≤10 subcategories each) + 1 catch-all

Defined in `taxonomy.py` (imported below). Patterns are lowercase regex fragments matched against `Charge Type` and `Charge Description` **combined into one string**, not tried separately field-by-field. (An earlier version tried `Charge Type` to completion first and fell back to `Charge Description` only if nothing matched — that badly undercounted subcategories like "Ground", because a generic `Charge Type` value like "Freight" would match the catch-all "Line Haul / Base Transportation" pattern before ever looking at `Charge Description`, where the real detail — e.g. "Ground Commercial" — actually lives.) Order still matters within the combined text: more specific subcategories/categories are listed before generic ones.

Changes from the original 6-major version, made after review feedback:
- **Discounts & Credits split into two majors** (`Discounts`, `Credits`) — rate-based discounts (earned/grace/general) and post-hoc credits/adjustments are conceptually different things and were being reported as one blended number.
- **Chargeback / Reversal moved to `Administrative & Service Fees`** — a chargeback is a billing event, not a discount; it doesn't belong with rate-based discounts either.
- **`Line Haul` and `Transportation / Base Charge` merged into `Line Haul / Base Transportation`** — these were two labels for the same dimension (the base/line-haul transport charge itself), as opposed to `Ground` / `Domestic Air` / `Ocean Freight` / `International`, which are transport mode / service type — a different dimension. Keeping them as six flat siblings made it look like "line haul" was a mode alongside ground/air/ocean, which it isn't.
- **New `Commercial / Business Delivery Surcharge` subcategory** under Accessorial, symmetric with the existing `Residential Delivery/Surcharge` — raw data has an equally large family of `... Commercial` labels (DAS Commercial, DAS Extended Commercial, DAS AK Commercial, etc.) that previously had no dedicated bucket.
- **`Billing Adjustment / Correction` moved from `Credits` to `Administrative & Service Fees`** — a row-level check (not just the aggregated total) found 99.1% of these 5,356 rows are *positive* charges (e.g. "Shipping Charge Correction Ground" is 4,376 positive vs. 6 negative rows), i.e. this is a billing/accounting correction that adds to the invoice, not a price concession (`Discounts`) or a carrier-issued refund (`Credits`). `Credits` now holds only `Credit From Carrier` (7 rows) — kept as its own major anyway since it's conceptually distinct (money the carrier actually returns) even though the current sample is small.

In [3]:
from taxonomy import TAXONOMY, CATCH_ALL, PRIORITY_OVERRIDES, TFIDF_EXCLUDE, normalize, rule_classify_row

n_major = len(TAXONOMY) + 1  # + catch-all
print(f"{n_major} major categories (incl. catch-all)")
for major, subcats in TAXONOMY.items():
    print(f"  {major}: {len(subcats)} subcategories")

8 major categories (incl. catch-all)
  Fuel Surcharge: 4 subcategories
  Accessorial / Delivery Surcharge: 10 subcategories
  Discounts: 3 subcategories
  Credits: 1 subcategories
  Taxes & Customs: 5 subcategories
  Administrative & Service Fees: 10 subcategories
  Line Haul / Base Transportation: 5 subcategories


## Rule-based classifier

`normalize` and `rule_classify_row` are imported from `taxonomy.py` above — `rule_classify_row` checks `PRIORITY_OVERRIDES` first, then falls through the `TAXONOMY` dict major-by-major, subcategory-by-subcategory, returning the first pattern match.

## Apply to unique (Charge Type, Charge Description) combinations

Classifying at the unique-combination level (not per row) is what keeps this fast and reusable at any data volume.

In [4]:
unique_labels = (
    detail.groupby(["Charge Type", "Charge Description"], dropna=False)["Charge Value"]
    .agg(row_count="size", total_value="sum")
    .reset_index()
)
print(f"{len(unique_labels):,} unique (Charge Type, Charge Description) combinations covering {unique_labels['row_count'].sum():,} rows")

classified = unique_labels.apply(lambda r: rule_classify_row(r["Charge Type"], r["Charge Description"]), axis=1)
unique_labels["Major Category"] = [c[0] if c else None for c in classified]
unique_labels["Subcategory"] = [c[1] if c else None for c in classified]
unique_labels["Method"] = ["rule" if c else None for c in classified]

matched_rows = unique_labels.loc[unique_labels["Major Category"].notna(), "row_count"].sum()
print(f"Rule-based coverage: {matched_rows / unique_labels['row_count'].sum() * 100:.1f}% of rows")

3,032 unique (Charge Type, Charge Description) combinations covering 256,764 rows


Rule-based coverage: 96.3% of rows


## TF-IDF fallback for labels the rules missed

For any (Charge Type, Charge Description) combo still unclassified, compare it against a small reference "document" built from each subcategory's own keywords, using character n-gram TF-IDF (robust to typos, pluralization, and non-English spellings) + cosine similarity. Assign to the best match if similarity clears a threshold, otherwise leave it in the catch-all bucket.

**Caveat found during QA** (see `charge_categorization_check.ipynb`): this fallback can score a *wrong* match confidently (0.4–0.7 similarity, well above the 0.25 threshold) when a label shares one generic word with a reference doc — e.g. "International Processing Fee" matched "International Fuel Surcharge" at 0.66 similarity purely because both strings contain "international", even though the charge has nothing to do with fuel. A handful of these were caught and moved into explicit rules above (`international processing`, `custom(s)? clearance`, `documents? preparation`, `eei`), but any *new* fallback match should still be treated as a suggestion to review, not an auto-accepted answer — a high similarity score alone doesn't guarantee it's correct.

**Update**: `Chargeback / Reversal` is excluded from this fallback's candidate pool (`TFIDF_EXCLUDE` in `taxonomy.py`) after finding it had a 100% false-positive rate there — its reference doc is just the single word "chargeback", which shares its first 6 characters with the very common substring "charge", so generic `"___ Charge"` labels ("Service Charge", "Weekend Charge", "Order Lane Charge", ...) were fuzzy-matching into it. Every literal "chargeback" mention in the data is already caught exactly by `PRIORITY_OVERRIDES`, so nothing real was lost by excluding it.

In [5]:
ref_docs, ref_labels = [], []
for major, subcats in TAXONOMY.items():
    for sub, patterns in subcats.items():
        if (major, sub) in TFIDF_EXCLUDE:
            continue
        doc = " ".join(p.replace(r"\b", "").replace(".*", " ").replace("'?", "").replace("?", "") for p in patterns)
        ref_docs.append(doc)
        ref_labels.append((major, sub))

vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5))
ref_vectors = vectorizer.fit_transform(ref_docs)

SIMILARITY_THRESHOLD = 0.25


def fallback_classify_row(charge_type, charge_desc):
    t = (normalize(charge_type) + " " + normalize(charge_desc)).strip()
    if not t:
        return None
    vec = vectorizer.transform([t])
    sims = cosine_similarity(vec, ref_vectors)[0]
    best_idx = sims.argmax()
    if sims[best_idx] >= SIMILARITY_THRESHOLD:
        return ref_labels[best_idx]
    return None


unmatched = unique_labels["Major Category"].isna()
for idx in unique_labels.index[unmatched]:
    row = unique_labels.loc[idx]
    result = fallback_classify_row(row["Charge Type"], row["Charge Description"])
    if result:
        unique_labels.loc[idx, "Major Category"] = result[0]
        unique_labels.loc[idx, "Subcategory"] = result[1]
        unique_labels.loc[idx, "Method"] = "tfidf_fallback"

unique_labels["Major Category"] = unique_labels["Major Category"].fillna(CATCH_ALL[0])
unique_labels["Subcategory"] = unique_labels["Subcategory"].fillna(CATCH_ALL[1])
unique_labels["Method"] = unique_labels["Method"].fillna("none")

final_coverage = 1 - unique_labels.loc[unique_labels["Major Category"] == CATCH_ALL[0], "row_count"].sum() / unique_labels["row_count"].sum()
print(f"Final coverage after TF-IDF fallback: {final_coverage * 100:.1f}% of rows categorized")
print(unique_labels["Method"].value_counts())

Final coverage after TF-IDF fallback: 98.4% of rows categorized
Method
rule              1499
none              1009
tfidf_fallback     524
Name: count, dtype: int64


## Summary: rows and $ value by category

In [6]:
summary = (
    unique_labels.groupby(["Major Category", "Subcategory"])
    .agg(row_count=("row_count", "sum"), total_value=("total_value", "sum"))
    .sort_values("row_count", ascending=False)
)
summary

row_count  total_value
Major Category                   Subcategory                                                     
Fuel Surcharge                   Domestic Fuel Surcharge                       67043    651965.62
Line Haul / Base Transportation  General / Mode Not Specified                  43266   6114412.42
Discounts                        General Discount                              36381  -3563696.20
                                 Earned Discount                               22904   -375705.98
Line Haul / Base Transportation  Ground                                        18068    256638.56
Discounts                        Grace Discount                                11814   -228506.64
Accessorial / Delivery Surcharge Delivery Area Surcharge (DAS)                  6755     95012.66
Fuel Surcharge                   Fuel Surcharge Correction                      5571     17309.02
Administrative & Service Fees    Billing Adjustment / Correction                5385     40890.80
Line Haul / Base Transportation  International / Export / Import Freight        4962    375047.25
Accessorial / Delivery Surcharge Residential Delivery/Surcharge                 4722     23743.67
Taxes & Customs                  VAT                                            4279     54524.71
Other / Uncategorized            Unclassified                                   4035    634599.24
Accessorial / Delivery Surcharge Wait Time / Mileage / Toll                     2693    110788.34
Administrative & Service Fees    Sustainability / Carbon Fee                    2409     13016.75
Line Haul / Base Transportation  Air Freight                                    2053    798321.42
Accessorial / Delivery Surcharge Additional Handling                            1798    148881.46
Taxes & Customs                  Customs / Brokerage                            1478     81031.36
Accessorial / Delivery Surcharge Demand / Peak Surcharge                        1401     53689.47
Fuel Surcharge                   International Fuel Surcharge                   1119      9857.14
Taxes & Customs                  GST / HST                                      1065     28338.49
Administrative & Service Fees    Third Party Billing                             918     15163.44
Accessorial / Delivery Surcharge Address Correction                              848     12477.50
                                 Signature Required                              803      6718.75
                                 Saturday / After Hours / Holiday                734     14587.59
Administrative & Service Fees    Pickup Service                                  654     71669.49
Taxes & Customs                  Duty & Import Tax                               578    169600.09
Administrative & Service Fees    Disbursement Fee                                554     13059.50
                                 Chargeback / Reversal                           481      6678.52
Accessorial / Delivery Surcharge Security Surcharge                              416     18248.24
Fuel Surcharge                   Chargeback Fuel Surcharge                       340      1108.82
Administrative & Service Fees    Document Fee                                    338     11848.44
                                 Returns / Print Label Fee                       334      5983.01
                                 Declared Value / Insurance                      229      9805.41
Taxes & Customs                  Sales / General Tax                             198     22185.60
Administrative & Service Fees    Package Handling / Storage                      104     46305.92
Line Haul / Base Transportation  Ocean Freight                                    23     13704.30
Credits                          Credit From Carrier                               7     -1695.99
Accessorial / Delivery Surcharge Commercial / Business Delivery Surcharge          4       116.16

In [7]:
major_summary = (
    unique_labels.groupby("Major Category")
    .agg(row_count=("row_count", "sum"), total_value=("total_value", "sum"))
    .sort_values("row_count", ascending=False)
)
major_summary["pct_of_rows"] = (major_summary["row_count"] / major_summary["row_count"].sum() * 100).round(1)
major_summary

,row_count,total_value,pct_of_rows
Major Category,,,
Fuel Surcharge,74073,680240.60,28.8
Discounts,71099,-4167908.82,27.7
Line Haul / Base Transportation,68372,7558123.95,26.6
Accessorial / Delivery Surcharge,20174,484263.84,7.9
Administrative & Service Fees,11406,234421.28,4.4
Taxes & Customs,7598,355680.25,3.0
Other / Uncategorized,4035,634599.24,1.6
Credits,7,-1695.99,0.0


## Review the catch-all bucket

Anything left in "Other / Uncategorized" is worth a manual look — either add a new keyword pattern to `TAXONOMY`, or it genuinely doesn't fit the taxonomy yet.

In [8]:
uncategorized = unique_labels[unique_labels["Major Category"] == CATCH_ALL[0]].sort_values("row_count", ascending=False)
print(f"{len(uncategorized):,} uncategorized combinations, {uncategorized['row_count'].sum():,} rows")
uncategorized.head(30)

1,009 uncategorized combinations, 4,035 rows


,Charge Type,Charge Description,row_count,total_value,Major Category,Subcategory,Method
2616,Pps Critical (American Airlines),Pps Critical (American Airlines),308,26965.40,Other / Uncategorized,Unclassified,none
2559,PREMIUM 12:00,PREMIUM 12:00,235,1175.00,Other / Uncategorized,Unclassified,none
1013,Entry Prep Fee,Entry Prep Fee,122,5882.59,Other / Uncategorized,Unclassified,none
680,CONTRACT MINIMUM REACHED MCADJ ADJ,CONTRACT MINIMUM REACHED MCADJ ADJ,122,9482.55,Other / Uncategorized,Unclassified,none
2580,Performance Pricing,Performance Pricing,118,-2522.82,Other / Uncategorized,Unclassified,none
2455,Order Lane Charge,Order Lane Charge,116,51543.76,Other / Uncategorized,Unclassified,none
2802,Service Charge,Service Charge,90,3469.00,Other / Uncategorized,Unclassified,none
2674,Retourzendingen 3 UPS Ophaalpogingen,Retourzendingen 3 UPS Ophaalpogingen,67,794.20,Other / Uncategorized,Unclassified,none
2833,Standard to Canada,Standard to Canada,63,2662.05,Other / Uncategorized,Unclassified,none
414,Accessorial,LIMITED ACCESS CHARGE FLAT,57,3067.74,Other / Uncategorized,Unclassified,none


## Save the reusable mapping table

This table is the reusable artifact: to classify a new/larger dataset, get its distinct (Charge Type, Charge Description) pairs, look them up here (or re-run `rule_classify_row` / `fallback_classify_label` on whatever's new), and merge the result back onto the full table — the expensive part (classification) never has to run per-row.

In [9]:
unique_labels.to_csv("outputs/charge_category_mapping.csv", index=False)
print("Saved outputs/charge_category_mapping.csv")

Saved outputs/charge_category_mapping.csv
